# MAISI-based CT generation pipeline testing

The purpose of this notebook is to test a modular CT generation pipeline based on the MAISI model. It loads the required VAE and Rectified Flow model components, prepares test patient CT data, encodes CT scans into latent representations, generates example CT variants, and saves the outputs for later evaluation.

This notebook focuses on validating the testing pipeline structure rather than model fine-tuning.

In [1]:
import os
from pathlib import Path

# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\Wiktoria\Documents\GitHub\robust-radiotherapy-planning


In [2]:
from src.pipeline.config import MaisiTestingConfig
from src.pipeline.data.prepare_test_data import prepare_test_data
from src.pipeline.evaluation.metrics import evaluate_generated_cts
from src.pipeline.inference.encode_latents import encode_latents
from src.pipeline.inference.run_generation import generate_ct_variants

In [3]:
config = MaisiTestingConfig(
    evaluation_split="val",
    cts_per_patient=2,
    use_dose_metrics=True,  # parent switch for every dose workflow
    use_base_dose_smoke_test=True,
    use_scenario_robustness=False,
    use_original_anatomy_comparison=False,
)

In [4]:
prepare_test_data(config)

Preparing CTs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [02:30<00:00,  1.87it/s]


In [5]:
encode_latents(config)

Encoding planning CTs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:38<00:00,  3.46s/it]


In [6]:
generate_ct_variants(config)

Generating CT variants: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [04:53<00:00, 26.70s/it]


In [7]:
evaluate_generated_cts(config)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


C:\Users\couch\Documents\robust-radiotherapy-planning\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\couch\Documents\robust-radiotherapy-planning\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: C:\Users\couch\Documents\robust-radiotherapy-planning\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


Calculating generated vs real structure metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [2:04:09<00:00, 677.25s/it]
Skipping missing reference structure for Patient_30: src\data_full\STRUCTURES\Patient_30\Patient_30_fraction_4_.nii.gz
Skipping missing reference structure for Patient_37: src\data_full\STRUCTURES\Patient_37\Patient_37_fraction_4_.nii.gz
Skipping missing reference structure for Patient_53: src\data_full\STRUCTURES\Patient_53\Patient_53_fraction_16_.nii.gz
Skipping missing reference structure for Patient_53: src\data_full\STRUCTURES\Patient_53\Patient_53_fraction_29_.nii.gz
Calculating pairwise variety metrics: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

## Standalone: run only dose metrics

Run only the cells in this section when CT generation and the image/structure metrics have already completed. Do **not** rerun `prepare_test_data`, `encode_latents`, `generate_ct_variants`, or `evaluate_generated_cts`.

This section calls `evaluate_doses` directly. It does not calculate image similarity, LPIPS, pairwise variety, or structure-similarity metrics. Existing processed CTs, structures, doses, generated CTs, and persisted warped-mask caches are reused. Dose CSV files are saved in `dose_only_config.metrics_dir`.

In [6]:
import importlib
from pathlib import Path

import src.pipeline.evaluation.dose_metrics as dose_metrics_module
import src.pipeline.evaluation.metrics as metrics_module
import src.pipeline.helpers.helpers as dose_helpers_module
from src.pipeline.config import MaisiTestingConfig

# Reload edited dose code in dependency order when this notebook kernel has
# already imported an older version of the pipeline.
importlib.reload(dose_helpers_module)
importlib.reload(dose_metrics_module)
importlib.reload(metrics_module)
evaluate_doses = metrics_module.evaluate_doses

dose_only_config = MaisiTestingConfig(
    evaluation_split="test",  # existing processed and generated data are stored under test
    use_structure_metrics=False,  # explicitly disable the heavy structure-metric stage
    use_dose_metrics=True,
    use_base_dose_smoke_test=True,
    use_scenario_robustness=True,
    use_original_anatomy_comparison=True,
)

In [3]:
# This calls only the dose orchestrator; evaluate_generated_cts() is intentionally not used.
evaluate_doses(dose_only_config)

Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_10_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_11_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_12_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_13_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_14_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_15_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_16_.pt: no dose found
Skipping real-anatomy dose metrics for RESULTS\MAISI_TESTING\processed_ct\test\Patient_40_fraction_17_.pt: no dose found
Skipping real-anatomy dose metri